# Chapter 1 &mdash; Regular Patterns: Repetition Without Counting

**Concept 10 of the Chapter 1 decomposition:** *Pattern Class I -- Regular Patterns*

Keywords, password rules, identifiers, comma lists. Unbounded repetition is fine; <b>remembering how much you repeated</b> is not.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Regular-Patterns/Concept-Regular-Patterns.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Two flavours:

* **finite and fixed-size** &mdash; the keyword `main` is four letters in that order;
* **finite but unbounded** &mdash; an identifier is a letter then any number of
  letters/digits; `keyword id, id, id;` is a keyword, then any number of ID+comma
  pairs, then `;`.

Regular patterns may repeat forever but **cannot count**. That single limitation is
what separates them from the next class.

## 2. Definitions

### The comma-list pattern as a DFA

`keyword id, id, id;` &mdash; here `k` is keyword, `i` is id, `c` is comma, `s` is semicolon.

In [ ]:
commalist = md2mc('''DFA
I   : k -> Ak
Ak  : i -> Ai
Ai  : c -> Ak
Ai  : s -> F
F   : k | i | c | s -> BH
I   : i | c | s -> BH
Ak  : k | c | s -> BH
Ai  : k | i -> BH
BH  : k | i | c | s -> BH
''')
print("comma-list DFA states :", sorted(commalist["Q"]))

### A "boring" repetition

$01001010010100101001\ldots$ is just an alternation of `01` and `001`. Boring is the
point: regular patterns repeat without memory of how often.

In [ ]:
boring = lstar({'01', '001'}, 3)
print("some strings over {01, 001} :", sorted(boring, key=len)[:10])

## 3. Tests

Any number of ID+comma pairs is accepted &mdash; **unbounded, but never counted**.

In [ ]:
for n in range(1, 7):
    s = 'k' + 'ic' * (n-1) + 'is'
    print("%d ids : %-18s accepted? %s" % (n, s, accepts_dfa(commalist, s)))
assert all(accepts_dfa(commalist, "k" + "ic"*(n-1) + "is") for n in range(1, 20))

Malformed declarations are rejected &mdash; the book's `keyword ; id id id,,` case.

In [ ]:
for s in ['ksiiicc', 'kis', 'kiciis', 'ks', 'kicis']:
    print("%-10s accepted? %s" % (s, accepts_dfa(commalist, s)))
assert not accepts_dfa(commalist, "ksiiicc")

The machine stays the **same size** however long the input is. That is the signature
of a regular pattern.

In [ ]:
print("states in the DFA        :", len(commalist["Q"]))
print("longest input we accepted:", len('k' + 'ic'*49 + 'is'))
print("accepted?                :", accepts_dfa(commalist, 'k' + 'ic'*49 + 'is'))
print()
print("50 ids, still 6 states. The machine never counted them.")

## 4. Animation


Watch the DFA cycle between "expecting an id" and "expecting a comma or semicolon".
The cycle is the unbounded repetition.


*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(commalist, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Design a DFA for the password rule "between 4 and 12 characters". How many states?
   Why is this still *finite and fixed-size*?
2. Modify `commalist` to allow an **empty** declaration `keyword ;`.
3. Try to design a DFA for "the same number of `(` as `)`". Where does it go wrong?
   (That is Concept 11.)

In [ ]:
# Your work for the exercises above.